# nano-dsv4.1f — Kaggle TPU v5e-8

Runnable smoke notebook for the public [`nano-dsv4.1f`](https://github.com/xiayicheng3-code/nano-dsv4.1f) repo.

It prioritizes **HBM safety and a modest 32K softmax** over tokenizer-efficiency experiments. You can run it from a repo checkout, or from a blank Kaggle notebook with Internet enabled—the bootstrap cell will clone the repo automatically.

For reproducible benchmarking, set `NANO_DSV41F_REF` to a branch, tag, or commit before the bootstrap cell. The exact checked-out commit is always printed.

In [ ]:
# TPU preflight first: do not modify Kaggle's JAX/libtpu environment.
import jax
import jax.numpy as jnp

print("JAX:", jax.__version__)
print("devices:", jax.devices())
print("process_count:", jax.process_count(), "local_device_count:", jax.local_device_count())

if not jax.devices() or jax.devices()[0].platform != "tpu":
    raise RuntimeError(
        "Select TPU in Kaggle Settings > Accelerator, restart the session, then rerun."
    )

## Repository bootstrap

If the source tree is already present, this cell uses it as-is. Otherwise it clones the public GitHub repo into `/kaggle/working/nano-dsv4.1f`.

Kaggle must have **Internet enabled** for the first clone. To pin a reproducible revision, set for example:

```python
import os
os.environ["NANO_DSV41F_REF"] = "<commit-or-tag>"
```

before running the cell.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/xiayicheng3-code/nano-dsv4.1f.git"
REPO_REF = os.environ.get("NANO_DSV41F_REF", "main")
TARGET = Path("/kaggle/working/nano-dsv4.1f")

def is_repo_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "nano_dsv41f").exists()

candidates = [Path.cwd(), Path.cwd().parent, TARGET]
root = next((p.resolve() for p in candidates if is_repo_root(p)), None)

if root is None:
    if TARGET.exists():
        raise RuntimeError(
            f"{TARGET} exists but is not a valid nano-dsv4.1f checkout. "
            "Remove/rename it, then rerun this cell."
        )
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(TARGET)],
            check=True,
        )
        root = TARGET
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Could not clone the public repo. In Kaggle, enable Internet in notebook "
            "settings and rerun. If Internet must stay off, attach a repo snapshot as "
            "a Kaggle Dataset or use the self-contained notebook export."
        ) from exc

# If a revision other than main was requested, fetch/check it out explicitly.
if REPO_REF != "main":
    try:
        subprocess.run(
            ["git", "fetch", "--depth", "1", "origin", REPO_REF],
            cwd=root,
            check=True,
        )
        subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=root, check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"Could not fetch/checkout NANO_DSV41F_REF={REPO_REF!r}") from exc

os.chdir(root)
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=root, text=True
).strip()
status = subprocess.check_output(
    ["git", "status", "--short"], cwd=root, text=True
).strip()

print("repo root:", root)
print("requested ref:", REPO_REF)
print("commit:", commit)
print("working tree:", "clean" if not status else "\n" + status)

In [ ]:
%pip install -q -e . --no-deps
print("installed nano-dsv41f without changing Kaggle JAX/libtpu")

## v5e-8 topology and conservative model defaults

In [ ]:
from nano_dsv41f import (
    ModelConfig, TrainConfig, V5E, make_v5e_mesh, runtime_report,
    semantic_axes, validate_sequence_length, validate_v5e_runtime,
)

config = ModelConfig()
train_config = TrainConfig(seq_len=4096)

print(runtime_report())
for warning in validate_v5e_runtime():
    print("WARNING:", warning)

mesh = make_v5e_mesh()
print("mesh:", mesh)
print("v5e:", V5E)
print("vocab_size:", config.vocab_size)
print("axes:", semantic_axes(config, mesh))
for warning in validate_sequence_length(train_config.seq_len, config, mesh):
    print("WARNING:", warning)

## Direct-to-shard BF16 initialization

In [ ]:
from nano_dsv41f import (
    init_model_sharded_mixed_precision,
    init_optimizer_state_sharded,
    memory_report,
    precision_summary,
)

params, param_specs, param_shardings = init_model_sharded_mixed_precision(
    jax.random.PRNGKey(0), config, mesh, payload_dtype=jnp.bfloat16
)
jax.block_until_ready(jax.tree_util.tree_leaves(params)[0])
print("dtypes:", precision_summary(params))
print("memory:", memory_report(params, param_specs, mesh))

opt_state, opt_state_shardings = init_optimizer_state_sharded(
    params, param_specs, config, mesh
)
jax.block_until_ready(jax.tree_util.tree_leaves(opt_state)[0])
print("optimizer state initialized")

## Compile static base / late-indexer executables

In [ ]:
from nano_dsv41f import compile_pretrain_step

base_step = compile_pretrain_step(
    params, opt_state, param_specs, config, train_config, mesh,
    include_indexer=False, n_segments=None,
)
late_indexer_step = compile_pretrain_step(
    params, opt_state, param_specs, config, train_config, mesh,
    include_indexer=True, n_segments=1,
)
print("created base_step and late_indexer_step")

## Synthetic `T=1024` smoke batch

`T=1024` gives 128 query tokens per chip under 8-way context sharding. This is intentionally smaller than the default 4096 training sequence so the first hardware check is conservative.

In [ ]:
import numpy as np
from nano_dsv41f import put_training_batch

smoke_t = 1024
host_ids = np.arange(smoke_t, dtype=np.int32)[None, :] % config.vocab_size
host_segments = np.zeros_like(host_ids, dtype=np.int32)
host_mask = np.ones_like(host_ids, dtype=bool)

ids, segments, token_mask = put_training_batch(
    host_ids, host_segments, host_mask, config, mesh
)
print("input sharding:", ids.sharding)
print("local shard:", ids.addressable_shards[0].data.shape)

## Compile diagnostics before execution

In [ ]:
from nano_dsv41f import compile_diagnostics

zero_step = jnp.asarray(0, jnp.int32)
compiled_base, diag = compile_diagnostics(
    base_step, params, opt_state, ids, segments, zero_step, token_mask
)

print("collectives:", diag["collectives"])
print("compiler memory:", diag["memory"])
print({
    k: v for k, v in diag["cost"].items()
    if any(t in k.lower() for t in ("flop", "byte", "transcend"))
})

## Standalone sharded Splash local-MQA parity

In [ ]:
from nano_dsv41f.splash import (
    dense_local_mqa_reference,
    make_v5e_sharded_local_mqa,
)

kq, kk = jax.random.split(jax.random.PRNGKey(7))
q = jax.random.normal(
    kq,
    (1, smoke_t, config.attention.n_heads, config.attention.head_dim),
    dtype=jnp.bfloat16,
)
kv = jax.random.normal(
    kk,
    (1, smoke_t, config.attention.head_dim),
    dtype=jnp.bfloat16,
)
seg = jnp.zeros((1, smoke_t), jnp.int32)

fn = make_v5e_sharded_local_mqa(
    mesh,
    seq_len=smoke_t,
    n_heads=config.attention.n_heads,
    head_dim=config.attention.head_dim,
    local_window=config.attention.local_window,
)

splash_out, splash_lse = jax.jit(fn)(q, kv, seg)
ref_out, ref_lse = jax.jit(
    lambda q, kv, s: dense_local_mqa_reference(
        q, kv, s, local_window=config.attention.local_window
    )
)(q, kv, seg)
jax.block_until_ready(splash_out)

print(
    "output max error:",
    float(jnp.max(jnp.abs(
        splash_out.astype(jnp.float32) - ref_out.astype(jnp.float32)
    ))),
)
print("LSE max error:", float(jnp.max(jnp.abs(splash_lse - ref_lse))))

## First training step

In [ ]:
params, opt_state, metrics = base_step(
    params, opt_state, ids, segments, zero_step, token_mask
)
jax.block_until_ready(metrics["loss"])
print({
    k: float(v)
    for k, v in metrics.items()
    if getattr(v, "ndim", 1) == 0
})

## Next

If the step is finite and HBM is comfortable, attach a pretrained ~32K tokenizer and packed data pipeline. Scale real tokens only after this smoke test; then use profiler evidence to choose between attention-kernel work and MoE dispatch work.

For any performance number you record, keep the printed **Git commit SHA**, Kaggle TPU type, sequence length, and compiler memory report with it.